# Notebook 05: Unsupervised Job and Skill Clustering
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Discover latent job archetypes and skill groupings without ground-truth labels.
2. Input text: composite representation (`job_title + job_description + requirements`).
3. Embeddings: SBERT Multilingual dense vectors (384 dimensions).
4. Run 4 clustering algorithm families:
   - K-Means across cluster counts $K \in [3, 4, 5, 6, 8, 10]$
   - Agglomerative Hierarchical Clustering (Cosine metric)
   - DBSCAN (density-based clustering with noise identification)
   - Gaussian Mixture Model (GMM on PCA-reduced dimensions)
5. Evaluate using Silhouette Score (Cosine), Davies-Bouldin Index, and Calinski-Harabasz Score.
6. Extract top discriminating keywords and assign human-interpretable cluster profiles.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.features.text_processor import prepare_clustering_text
from src.models.clustering import run_clustering_experiments

## 1. Load Clustering Dataset and Composite Embeddings
We load `clustering_data.csv` and the corresponding `sbert_clustering_embeddings.npy` matrix.

In [ ]:
df_cluster = pd.read_csv(config.PROCESSED_DATA_DIR / "clustering_data.csv")
cluster_embeddings = np.load(config.EMBEDDINGS_DIR / "sbert_clustering_embeddings.npy")
composite_texts, _ = prepare_clustering_text(df_cluster)

print(f"Clustering samples: {cluster_embeddings.shape[0]}, Dimensions: {cluster_embeddings.shape[1]}")
print(f"Sample composite text (first 150 chars):\n{composite_texts[0][:150]}...")

## 2. Execute Clustering Algorithms and Evaluation
Compute clustering metrics across K-Means, Agglomerative, DBSCAN, and GMM.

In [ ]:
clustering_results, assignments, best_km_model, metadata = run_clustering_experiments(
    df_cluster, cluster_embeddings, composite_texts
)

print("=== Clustering Evaluation Metrics Comparison ===")
display(clustering_results) if 'display' in dir() else print(clustering_results)

## 3. Discovered Cluster Profiles and Top Keywords
Inspect the interpreted cluster archetypes, their top 10 keywords, and job volumes.

In [ ]:
kw_path = config.CLUSTERING_OUTPUTS / 'metrics' / 'cluster_keywords.csv'
if kw_path.exists():
    kw_df = pd.read_csv(kw_path)
    print("=== Discovered Cluster Profiles & Keywords ===")
    display(kw_df) if 'display' in dir() else print(kw_df.to_string(index=False))

from IPython.display import Image, display as ipy_display

for plot_name in ['elbow_curve.png', 'silhouette_comparison.png', 'pca_clusters.png', 'cluster_distribution.png']:
    p = config.CLUSTERING_OUTPUTS / 'plots' / plot_name
    if p.exists():
        print(f"\n--- {plot_name} ---")
        ipy_display(Image(filename=str(p)))